## Week-2

In [2]:
import pandas as pd

df = pd.read_csv("CRMLSListing_residential.csv")
df.head(5)

/var/folders/d_/gll3jlg1445_2nm3xgq_xvdc0000gn/T/ipykernel_3085/704966105.py:3: DtypeWarning: Columns (0: ListAgentEmail, 1: BuyerAgencyCompensationType) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("CRMLSListing_residential.csv")


,OriginalListPrice,ListingKey,ListAgentEmail,CloseDate,ClosePrice,ListAgentFirstName,ListAgentLastName,Latitude,Longitude,UnparsedAddress,...,NewConstructionYN,GarageSpaces,HighSchoolDistrict,PostalCode,BuyerOfficeName.1,AssociationFee,LotSizeSquareFeet,MiddleOrJuniorSchoolDistrict,UnparsedAddress.1,source_file
0,1340000.0,1074973329,haleh360@Gmail.com,NaN,NaN,Haleh,Dowlatshahi,34.052207,-118.408445,2220 Avenue Of The Stars 2704,...,False,NaN,NaN,90067,NaN,2105.00,177861.0,NaN,2220 Avenue Of The Stars 2704,CRMLSListing202401.csv
1,2500000.0,1074954552,Reneechen@yourhomesoldguaranteed.com,NaN,NaN,Renee,Chen,33.496363,-117.691677,16 Palisades,...,False,3.0,Capistrano Unified,92677,NaN,254.00,5300.0,NaN,16 Palisades,CRMLSListing202401.csv
2,3150000.0,1074936537,anader@dppre.com,NaN,NaN,Margaret,Nader,34.119345,-118.111254,1615 Waverly Road,...,NaN,2.0,NaN,91108,NaN,NaN,9404.0,NaN,1615 Waverly Road,CRMLSListing202401.csv
3,3090000.0,1074917818,QIANYU0607@GMAIL.COM,NaN,NaN,QIANYU,GUAN,33.984057,-117.802819,2250 Indian Creek Road,...,False,4.0,Walnut Valley Unified,91765,NaN,295.95,58232.0,NaN,2250 Indian Creek Road,CRMLSListing202401.csv
4,12725000.0,1074143166,jeff.williams@pacificsir.com,NaN,NaN,Jeff,Williams,33.607583,-117.887743,317 E. Bayfront,...,False,2.0,Newport Mesa Unified,92662,NaN,0.00,2250.0,NaN,317 E. Bayfront,CRMLSListing202401.csv


In [3]:
cat_cols = df.select_dtypes(include=["object", "string"]).columns
print("Categorical columns:", len(cat_cols))
print(cat_cols)


Categorical columns: 48
Index(['ListAgentEmail', 'CloseDate', 'ListAgentFirstName',
       'ListAgentLastName', 'UnparsedAddress', 'PropertyType',
       'ListOfficeName', 'BuyerOfficeName', 'CoListOfficeName',
       'ListAgentFullName', 'CoListAgentFirstName', 'CoListAgentLastName',
       'BuyerAgentMlsId', 'BuyerAgentFirstName', 'BuyerAgentLastName',
       'AssociationFeeFrequency', 'MLSAreaMajor', 'CountyOrParish',
       'PropertyType.1', 'MlsStatus', 'ElementarySchool',
       'ListAgentFirstName.1', 'AttachedGarageYN', 'BuilderName',
       'PropertySubType', 'SubdivisionName', 'BuyerOfficeAOR',
       'BuyerAgencyCompensationType', 'ListingId', 'City',
       'ContractStatusChangeDate', 'CoBuyerAgentFirstName',
       'PurchaseContractDate', 'ListingContractDate', 'StateOrProvince',
       'MiddleOrJuniorSchool', 'FireplaceYN', 'HighSchool', 'Levels',
       'ListAgentLastName.1', 'CloseDate.1', 'LotSizeDimensions',
       'NewConstructionYN', 'HighSchoolDistrict', 'PostalCod

In [4]:
import pandas as pd

# =========================
# 1. Basic Info
# =========================
print("=== DATA SHAPE ===")
print(df.shape)

# 2. Build categorical review table
# =========================
cat_review = pd.DataFrame({
    "column": cat_cols,
    "missing_count": [df[col].isna().sum() for col in cat_cols],
    "missing_pct": [round(df[col].isna().mean() * 100, 2) for col in cat_cols],
    "n_unique": [df[col].nunique(dropna=True) for col in cat_cols]
})

# =========================
# 3. Sample top values
# =========================
top_values = []
for col in cat_cols:
    vals = df[col].value_counts(dropna=False).head(5)
    vals_str = "; ".join([f"{str(idx)} ({cnt})" for idx, cnt in vals.items()])
    top_values.append(vals_str)

cat_review["top_values_sample"] = top_values

# =========================
# 4. Core columns
# =========================
core_columns = [
    "PropertyType", "PropertySubType", "City", "PostalCode",
    "CountyOrParish", "StateOrProvince", "MLSAreaMajor",
    "MlsStatus", "ListingId", "UnparsedAddress",
    "ListingContractDate", "ContractStatusChangeDate",
    "PurchaseContractDate", "OriginatingSystemName",
    "OriginatingSystemSubName"
]

# =========================
# 5. Decision
# =========================
def classify_column(row):
    col = row["column"]
    missing_pct = row["missing_pct"]
    n_unique = row["n_unique"]

    if col in core_columns:
        return pd.Series(["Keep (Core)"])
    elif missing_pct >= 90:
        return pd.Series(["Drop"])
    elif n_unique == 1:
        return pd.Series(["Drop"])
    elif "Name" in col or "Email" in col:
        return pd.Series(["Drop"])
    else:
        return pd.Series(["Review"])

cat_review[["decision"]] = cat_review.apply(classify_column, axis=1)

# =========================
# 6. Duplicate Check (overall)
# =========================
total_duplicates = df.duplicated().sum()
dup_pct = round(total_duplicates / len(df) * 100, 2)

print("\n=== DUPLICATE ROWS ===")
print(f"Total duplicate rows: {total_duplicates}")
print(f"Duplicate %: {dup_pct}%")

# =========================
# 7. Duplicate Check (key columns)
# =========================
print("\n=== DUPLICATE CHECK BY KEY COLUMNS ===")

if "ListingId" in df.columns:
    print("Duplicate ListingId:", df["ListingId"].duplicated().sum())

if "UnparsedAddress" in df.columns:
    print("Duplicate Address:", df["UnparsedAddress"].duplicated().sum())

if "latfilled" in df.columns and "lonfilled" in df.columns:
    print("Duplicate Geo:", df.duplicated(subset=["latfilled", "lonfilled"]).sum())

# =========================
# 8. Sort table
# =========================
cat_review = cat_review.sort_values(by="missing_pct", ascending=False).reset_index(drop=True)

# =========================
# 9. Display
# =========================
print("\n=== KEEP (CORE) ===")
display(cat_review[cat_review["decision"] == "Keep (Core)"])

print("\n=== DROP ===")
display(cat_review[cat_review["decision"] == "Drop"])

print("\n=== REVIEW ===")
display(cat_review[cat_review["decision"] == "Review"])

=== DATA SHAPE ===
(540183, 85)

=== DUPLICATE ROWS ===
Total duplicate rows: 0
Duplicate %: 0.0%

=== DUPLICATE CHECK BY KEY COLUMNS ===
Duplicate ListingId: 92
Duplicate Address: 65531

=== KEEP (CORE) ===


,column,missing_count,missing_pct,n_unique,top_values_sample,decision
20,PurchaseContractDate,273091,50.56,833,nan (273091); 2024-04-24 (880); 2024-05-01 (87...,Keep (Core)
23,MLSAreaMajor,73002,13.51,1107,nan (73002); 699 - Not Defined (61103); SRCAR ...,Keep (Core)
27,ContractStatusChangeDate,6020,1.11,848,nan (6020); 2024-04-30 (2372); 2024-04-26 (220...,Keep (Core)
30,PropertySubType,1247,0.23,21,SingleFamilyResidence (394000); Condominium (9...,Keep (Core)
32,UnparsedAddress,670,0.12,474651,nan (670); 5166 Via Portola (12); 1027 S Lucer...,Keep (Core)
33,City,584,0.11,1209,Los Angeles (27865); San Diego (25431); San Jo...,Keep (Core)
38,StateOrProvince,62,0.01,28,CA (539967); OS (85); nan (62); AZ (17); OR (7),Keep (Core)
39,CountyOrParish,0,0.00,63,Los Angeles (137288); Riverside (75727); San D...,Keep (Core)
40,ListingId,0,0.00,540091,NDP2501178 (4); PTP2400423 (2); PTP2400378 (2)...,Keep (Core)
43,PropertyType,0,0.00,1,Residential (540183),Keep (Core)



=== DROP ===


,column,missing_count,missing_pct,n_unique,top_values_sample,decision
0,CoBuyerAgentFirstName,525755,97.33,2790,nan (525755); Michael (199); Andrew (150); Ste...,Drop
1,BuilderName,514943,95.33,3798,nan (514943); Lennar (2357); Toll Brothers (12...,Drop
2,LotSizeDimensions,511990,94.78,11960,nan (511990); 50x135 (482); 50x100 (319); 50x1...,Drop
7,CoListAgentFirstName,419643,77.69,7450,nan (419643); Michael (1776); David (1611); Ka...,Drop
8,CoListAgentLastName,419242,77.61,15352,nan (419242); Myatt (1132); Smith (744); Flore...,Drop
9,CoListOfficeName,419157,77.60,6156,nan (419157); Compass (14081); Coldwell Banker...,Drop
11,BuyerOfficeName,383233,70.95,13801,nan (383233); Compass (12527); Coldwell Banker...,Drop
12,BuyerOfficeName.1,383233,70.95,13801,nan (383233); Compass (12527); Coldwell Banker...,Drop
13,BuyerAgentFirstName,381704,70.66,13792,nan (381704); General (2026); Michael (1728); ...,Drop
15,BuyerAgentLastName,380976,70.53,28169,nan (380976); NONMEMBER (2059); Nguyen (990); ...,Drop



=== REVIEW ===


,column,missing_count,missing_pct,n_unique,top_values_sample,decision
3,ElementarySchool,475845,88.09,1423,nan (475845); Other (13125); Harbor View (412)...,Review
4,MiddleOrJuniorSchool,475701,88.06,674,nan (475701); Other (9794); Rancho Santa Marga...,Review
5,BuyerAgencyCompensationType,461170,85.37,3,nan (461170); Item1 (77938); Item (969); SeeRe...,Review
6,HighSchool,455717,84.36,530,nan (455717); Other (4075); Tesoro (1252); Por...,Review
10,BuyerOfficeAOR,393468,72.84,66,nan (393468); SanDiego (11548); OrangeCounty (...,Review
14,BuyerAgentMlsId,381086,70.55,71586,nan (381086); NONMEMBER (2481); nonmember (865...,Review
16,CloseDate,374361,69.30,907,nan (374361); 2024-03-29 (1057); 2024-04-30 (1...,Review
17,CloseDate.1,374361,69.30,907,nan (374361); 2024-03-29 (1057); 2024-04-30 (1...,Review
19,AssociationFeeFrequency,309326,57.26,4,nan (309326); Monthly (216751); Annually (9502...,Review
21,HighSchoolDistrict,162408,30.07,451,nan (162408); Other (44528); Los Angeles Unifi...,Review


##Second Round of Screening: Review Phase

Drop:
BuyerAgencyCompensationType，BuyerOfficeAOR, BuyerAgentMlsId, AssociationFeeFrequency, FireplaceYN

Duplicates (Drop): CloseDate.1,UnparsedAddress.1,PropertyType.1

keep:
HighSchool- School District Housing

CloseDate- With for-sale listings, the date the purchase agreement was fulfilled. 

HighSchoolDistrict	- The name of the high school district having a catchment area that includes the a ssociated property. 

AttachedGarageYN- A marker used to indicate whether a garage is attached to the main residence.
Levels- the number of levels in the property being sold. 

NewConstructionYN-- Is the property newly constructed and has not been
previously occupied?






